In [ ]:
import numpy as np
import pandas as pd

rand = np.random.default_rng(0) # 랜덤값 생성
n = 3000    # 3000개 로트
classes = ['설비', '자재', '품질', '운영']
# classes 안에서 3000개의 데이터를 생성하는데 p 의 비율대로 생성하여 loss 에 담아라
loss = rand.choice(classes, size=n, p=[0.60, 0.22, 0.12, 0.06])
# print(loss[:10])
# 이 내용을 각각의 변수에 담는다.
is_eq, is_mat, is_q = (loss == '설비'), (loss == '자재'), (loss == '품질')
print(is_eq[:10]) # 설비로 기록된 row True 나머진 False
print(is_mat[:10]) # 자재로 기록된 row True 나머진 False
print(is_q[:10]) # 품질로 기록된 row True 나머진 False

[False  True  True  True False False False False  True False]
[ True False False False  True False  True  True False False]
[False False False False False  True False False False  True]


In [6]:
# 데이터 생산
df = pd.DataFrame({
    # poisson : 횟수나 건수같은 데이터를 만들 때 쓰는 포아송 분포(Poisson Distribution)
    'alarm_cnt'  : rand.poisson(np.where(is_eq, 4, 1)), # 설비고장시 알람 횟수
    # normal([data],3) : 정규분포, 표준편차 3
    'equip_temp' : rand.normal(np.where(is_eq, 68, 62), 4), # 설비 고장시 온도 
    'prev_ct'    : rand.normal(np.where(is_eq, 48, 45), 3), # 설비 고장시 직전 CT(초)
    # rand.binomial(n, p, size) :  시도횟수,성공확률, 데이터
    # 자재문제일 경우 - 입고확률 15%, 또는 입고확률 90% 로 True
    'material_in': np.where(is_mat, rand.binomial(1, 0.15, n),rand.binomial(1, 0.9, n)),                    
    'wip_buffer' : rand.poisson(np.where(is_mat, 2, 8)), # 자재문제일 경우 공정 대기 재공 수
    'ng_cnt'     : rand.poisson(np.where(is_q, 5, 1)), # 품질 문제일 경우 직전 불량 수
    'shift'      : rand.choice(['주간', '야간'], n),
    # rand.gamma(shape, scale) : 움수가 나오지 않는 오른쪽으로 치우친 형태의 데이터를 만들 때 사용하는 확률 분포
    # - shape:1 이하면 급격하게 솟아올랐다가 급격히 떨어지는 형태, 클수록 완만한 곡선
    # - scale : 평균값과 최댓값 자체가 커지게됨
    'duration'   : np.round(rand.gamma(2, np.where(is_eq, 15, np.where(is_mat, 10, 6))), 1),
    'loss_type'  : loss,
})
df.head()

,alarm_cnt,equip_temp,prev_ct,material_in,wip_buffer,ng_cnt,shift,duration,loss_type
0,1,65.391425,41.672671,1,4,1,주간,23.7,자재
1,2,67.343397,46.115500,0,8,3,주간,27.8,설비
2,7,68.489734,49.394085,1,10,1,주간,43.3,설비
3,6,72.852641,49.816136,1,8,1,주간,11.6,설비
4,0,58.639210,46.221190,0,5,3,주간,12.2,자재


In [8]:
df.to_csv('/content/data/stop_events.csv', index=False)